# 03 · Use Case Data Quality & Validation
Validates that all team engagements are properly linked to Salesforce use cases.
Identifies meetings not matched by Setsail (Snowflake's calendar-to-SF sync) and accounts needing use case creation.

In [ ]:
import datetime, json
import matplotlib.pyplot as plt, matplotlib.patches as mpatches, matplotlib.gridspec as gs
import pandas as pd
from IPython.display import display, HTML
plt.switch_backend("agg")

SF_BLUE="#29B5E8"; SF_TEAL="#00A9CE"; SF_GREEN="#36B37E"
SF_ORANGE="#FF8B00"; SF_DARK="#0A2859"; SF_GRAY="#D0D0D0"; BG="#F9FAFB"

def clean(ax):
    ax.set_facecolor(BG)
    for s in ax.spines.values(): s.set_visible(False)

def fmt_eacv(v):
    if not v or (isinstance(v,float) and str(v)=="nan"): return "$0"
    v=float(v)
    return f"${v/1e6:.1f}M" if v>=1e6 else f"${v/1e3:.0f}K" if v>=1e3 else f"${v:.0f}"

def html_table(df, row_color_fn=None):
    if hasattr(df, 'to_pandas'): df = df.to_pandas()
    html = "<table style='border-collapse:collapse;width:100%;font-size:13px;font-family:sans-serif'>"
    html += "<tr style='background:#0A2859;color:white'>" + "".join("<th style='padding:8px 12px;text-align:left'>" + str(col) + "</th>" for col in df.columns) + "</tr>"
    for i, (_, r) in enumerate(df.iterrows()):
        bg = row_color_fn(r) if row_color_fn else ("#f8f9fa" if i%2==0 else "white")
        html += "<tr style='background:" + bg + "'>" + "".join("<td style='padding:7px 12px'>" + str(v if v is not None else '') + "</td>" for v in r) + "</tr>"
    html += "</table>"
    display(HTML(html))

TEAM = {
    "Jim Lebonitte": "005VI00000Z0y6bYAB",
    "Michael Hamilton": "005VI00000ExC3VYAV",
    "Patrick Sheehan": "0053r00000BblkZAAR",
    "Phani Alapaty": "005VI00000HajknYAB",
    "Steve Mitchener": "0050Z000009IrKRQA0",
    "Zaki Bajwa": "005VI00000XibQfYAJ"
}
TEAM_IDS   = list(TEAM.values())
team_ids_str = "', '".join(TEAM_IDS)
ACT_TABLE  = "TEMP.JLEBONITTE_EDA_ACTIVITY_TRACKING.SE_WEEKLY_ACTIVITIES"

fy_start   = datetime.date(2026, 2, 1)
q2_start   = datetime.date(2026, 5, 1)
today      = datetime.date.today()
fy_start_str = str(fy_start)
q2_start_str = str(q2_start)
today_str    = str(today)
print("Setup complete ✓")

# ─── FILTERS ─────────────────────────────────────────────
period_start = datetime.date(2026, 4, 1)
period_end   = today
# ─────────────────────────────────────────────────────────
ps, pe = str(period_start), str(period_end)
print(f"Data quality check  |  {ps} → {pe}")


## Overall Data Quality Scorecard

In [ ]:
%%sql -r dq_scorecard
SELECT
    COUNT(*)                                                                     AS total_rows,
    SUM(IFF(SF_ACCOUNT_ID IS NOT NULL,1,0))                                      AS has_account_id,
    ROUND(100.0*SUM(IFF(SF_ACCOUNT_ID IS NOT NULL,1,0))/COUNT(*),1)             AS pct_account_id,
    SUM(IFF(SF_ACTIVITY_ID IS NOT NULL,1,0))                                     AS setsail_matched,
    ROUND(100.0*SUM(IFF(SF_ACTIVITY_ID IS NOT NULL,1,0))/COUNT(*),1)            AS setsail_match_pct,
    SUM(IFF(USE_CASE_TAGGED_IN_SF='Yes',1,0))                                    AS use_case_tagged,
    ROUND(100.0*SUM(IFF(USE_CASE_TAGGED_IN_SF='Yes',1,0))/NULLIF(SUM(IFF(SF_ACCOUNT_ID IS NOT NULL,1,0)),0),1) AS pct_uc_of_known_accounts,
    SUM(IFF(USE_CASE_TAGGED_IN_SF='No' AND SF_ACCOUNT_ID IS NOT NULL,1,0))      AS uc_gap_count,
    SUM(IFF(SF_ACCOUNT_ID IS NULL,1,0))                                          AS no_account_id
FROM TEMP.JLEBONITTE_EDA_ACTIVITY_TRACKING.SE_WEEKLY_ACTIVITIES
WHERE MEETING_DATE BETWEEN '{{ps}}' AND '{{pe}}'

In [ ]:
dq_scorecard = dq_scorecard.to_pandas() if hasattr(dq_scorecard, "to_pandas") else dq_scorecard
r = dq_scorecard.iloc[0]
metrics = [
    ("Account ID Resolved",     float(r.get("PCT_ACCOUNT_ID",0) or 0),  SF_BLUE),
    ("Setsail Matched",   float(r.get("SETSAIL_MATCH_PCT",0) or 0),      SF_TEAL),
    ("Use Case Coverage\n(of resolved accounts)", float(r.get("PCT_UC_OF_KNOWN_ACCOUNTS",0) or 0), SF_GREEN),
]
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
for ax, (title, val, color) in zip(axes, metrics):
    clean(ax)
    ax.pie([val, 100-val], colors=[color, SF_GRAY], startangle=90, wedgeprops={"linewidth":0})
    ax.text(0, 0, f"{val:.0f}%", ha="center", va="center", fontsize=20, fontweight="bold", color=SF_DARK)
    ax.set_title(title, color=SF_DARK, fontsize=11)
plt.suptitle(f"Data Quality Scorecard · {ps} → {pe}", color=SF_DARK, fontsize=12)
plt.tight_layout(); plt.show()
print(f"UC Gap: {r['UC_GAP_COUNT']} accounts need a use case created | Backlog: {r['TOTAL_ROWS']-int(r['SETSAIL_MATCHED'])} meetings not logged | No account: {r['NO_ACCOUNT_ID']}")

## Use Case Gap List — Accounts Met But No Use Case

In [ ]:
%%sql -r uc_gap
SELECT
    COALESCE(CUSTOMER,'(unknown)')  AS customer,
    SF_ACCOUNT_ID,
    MEETING_SE_NAME                 AS specialist,
    AE_NAME,
    AE_EMAIL,
    COUNT(*)                        AS meetings,
    MAX(MEETING_DATE)               AS last_touch,
    MAX(OPPORTUNITY_NAME)           AS opportunity
FROM TEMP.JLEBONITTE_EDA_ACTIVITY_TRACKING.SE_WEEKLY_ACTIVITIES
WHERE USE_CASE_TAGGED_IN_SF = 'No'
  AND SF_ACCOUNT_ID IS NOT NULL
  AND MEETING_DATE BETWEEN '{{ps}}' AND '{{pe}}'
GROUP BY 1,2,3,4,5
ORDER BY 6 DESC

In [ ]:
uc_gap = uc_gap.to_pandas() if hasattr(uc_gap, "to_pandas") else uc_gap
print(f"Accounts with meetings but no use case tagged: {len(uc_gap)}")
print("Reach out to the AE/SE to create a use case in Salesforce:\n")
html_table(uc_gap)

## Setsail Unmatched Meetings

In [ ]:
%%sql -r backlog
SELECT
    MEETING_DATE,
    COALESCE(CUSTOMER,'(unknown)')  AS customer,
    MEETING_TITLE,
    MEETING_SE_NAME                 AS specialist,
    SOURCE,
    SF_ACCOUNT_ID,
    OPPORTUNITY_NAME,
    USE_CASE_NAME
FROM TEMP.JLEBONITTE_EDA_ACTIVITY_TRACKING.SE_WEEKLY_ACTIVITIES
WHERE SF_ACTIVITY_ID IS NULL
  AND MEETING_DATE BETWEEN '{{ps}}' AND '{{pe}}'
ORDER BY MEETING_DATE DESC

In [ ]:
backlog = backlog.to_pandas() if hasattr(backlog, "to_pandas") else backlog
print(f"Meetings not matched by Setsail: {len(backlog)}")
html_table(backlog)

## Setsail Match Rate by Specialist

In [ ]:
%%sql -r backlog_by_se
SELECT
    COALESCE(MEETING_SE_NAME,'Unknown')  AS specialist,
    SUM(IFF(SF_ACTIVITY_ID IS NULL,1,0)) AS unlogged,
    COUNT(*)                              AS total,
    ROUND(100.0*SUM(IFF(SF_ACTIVITY_ID IS NULL,1,0))/COUNT(*),1) AS pct_unlogged
FROM TEMP.JLEBONITTE_EDA_ACTIVITY_TRACKING.SE_WEEKLY_ACTIVITIES
WHERE MEETING_DATE BETWEEN '{{ps}}' AND '{{pe}}'
GROUP BY 1 ORDER BY 2 DESC

In [ ]:
backlog_by_se = backlog_by_se.to_pandas() if hasattr(backlog_by_se, "to_pandas") else backlog_by_se
df = backlog_by_se
fig, ax = plt.subplots(figsize=(10,3.5)); clean(ax)
bars = ax.bar(df["SPECIALIST"], df["UNLOGGED"], color=SF_ORANGE, alpha=0.85)
ax.bar(df["SPECIALIST"], df["TOTAL"]-df["UNLOGGED"], bottom=df["UNLOGGED"], color=SF_BLUE, alpha=0.6, label="Logged")
for bar, pct in zip(bars, df["PCT_UNLOGGED"]):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()/2,
            f"{pct:.0f}%\nunlogged", ha="center", va="center", fontsize=9, color="white", fontweight="bold")
ax.set_title("Setsail Match Rate by Specialist", color=SF_DARK, fontsize=12)
ax.set_ylabel("Meetings")
ax.legend(["Unlogged","Logged"], frameon=False)
plt.tight_layout(); plt.show()

## Unknown Account IDs — Meetings We Couldn't Resolve

In [ ]:
%%sql -r unknown_accts
SELECT
    MEETING_TITLE,
    MEETING_DATE,
    MEETING_SE_NAME  AS specialist,
    SOURCE,
    ATTENDEES
FROM TEMP.JLEBONITTE_EDA_ACTIVITY_TRACKING.SE_WEEKLY_ACTIVITIES
WHERE SF_ACCOUNT_ID IS NULL
  AND MEETING_DATE BETWEEN '{{ps}}' AND '{{pe}}'
ORDER BY MEETING_DATE DESC LIMIT 30

In [ ]:
unknown_accts = unknown_accts.to_pandas() if hasattr(unknown_accts, "to_pandas") else unknown_accts
print(f"Meetings with no SF Account ID resolved: {len(unknown_accts)}")
html_table(unknown_accts)